# THINGS-EEG2 → CLIP: EEG concept decoding

**Cengiz Çakır · Computational neuroscience / machine learning portfolio project**

Map repetition-averaged EEG from subject 1 into frozen CLIP **text** embeddings and rank object concepts by cosine similarity. This is concept classification; the notebook does not reconstruct images.

Run Sections 0–6 in order. Optional training experiments are disabled by default. GPU recommended; CPU is supported but slow. Several GB of data and model weights are downloaded and cached. Metrics are produced when training completes; no measured results are bundled.

The original exploratory notebook is organized here into a reproducible workflow, with concept-disjoint validation, explicit data checks, and separate models for the two evaluation protocols.

## 0. Setup & data loading

In [ ]:
# Install dependencies in this notebook's Python environment.
%pip install -q "numpy>=1.24,<2" "torch>=2.2,<3" "transformers>=4.40,<5" "scikit-learn>=1.3,<2" "matplotlib>=3.7,<4" "pandas>=2,<3" "requests>=2.31,<3" "osfclient>=0.0.5,<1"
import os, random, subprocess, json
from pathlib import Path
import numpy as np
import torch

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WORK = Path(os.environ.get("THINGS_EEG2_DATA_DIR", "/content/things_eeg2" if Path("/content").exists() else "../data"))
WORK.mkdir(parents=True, exist_ok=True)
PREP_PROJECT = "3eayd"
SUB = 1
RUN_BASELINE = False  # Optional 150-epoch comparison; excluded from the default workflow.
RUN_WITHIN_CONCEPT = False  # Optional fresh-model experiment.
RUN_DNN_EXPLORATION = False
RESULTS = WORK / "results"
RESULTS.mkdir(exist_ok=True)
print("device:", DEVICE, "| cache:", WORK.resolve())

def osf_ls(project):
    result = subprocess.run(["osf", "-p", project, "ls"], capture_output=True, text=True, check=True)
    return [line.strip() for line in result.stdout.splitlines() if line.strip()]

def osf_fetch(project, remote, local):
    local = Path(local)
    temporary = local.with_suffix(local.suffix + ".part")
    try:
        subprocess.run(["osf", "-p", project, "fetch", remote, str(temporary)], check=True)
        temporary.replace(local)
    finally:
        temporary.unlink(missing_ok=True)

def load_split(split):
    local = WORK / f"sub-{SUB:02d}_eeg_{split}.npy"
    if not local.exists():
        matches = [p for p in osf_ls(PREP_PROJECT)
                   if f"sub-{SUB:02d}" in p.lower() and split in p.lower() and p.lower().endswith(".npy")]
        if len(matches) != 1:
            raise RuntimeError(f"Expected one {split} file for subject {SUB}, found {matches}. Inspect osf_ls(PREP_PROJECT).")
        print("Downloading:", matches[0], "(large file; cached on later runs)")
        osf_fetch(PREP_PROJECT, matches[0], local)
    # Official dataset files contain pickled dictionaries: use only trusted OSF/local inputs.
    data = np.load(local, allow_pickle=True).item()
    eeg = np.asarray(data["preprocessed_eeg_data"])
    if eeg.ndim != 4:
        raise ValueError(f"Expected images × repetitions × channels × time, got {eeg.shape}")
    print(split, eeg.shape, eeg.dtype)
    return data

train = load_split("train")
test = load_split("test")
eeg, eeg_test = train["preprocessed_eeg_data"], test["preprocessed_eeg_data"]

### Image labels and preprocessing metadata
Read embedded metadata when available; otherwise download the original image metadata from the official THINGS-EEG2 image component. Label count checks guard against mismatched exports.

In [ ]:
import requests

# Original THINGS-EEG2 exports keep image labels in a separate metadata file.
if "img_metadata" not in train or "img_metadata" not in test:
    metadata_path = WORK / "image_metadata.npy"
    if not metadata_path.exists():
        response = requests.get("https://osf.io/download/qkgtf/", timeout=180)
        response.raise_for_status()
        temporary = metadata_path.with_suffix(".npy.part")
        temporary.write_bytes(response.content)
        temporary.replace(metadata_path)
    image_metadata = np.load(metadata_path, allow_pickle=True).item()
    train.setdefault("img_metadata", image_metadata)
    test.setdefault("img_metadata", image_metadata)
for split, data in [("train", train), ("test", test)]:
    labels = data["img_metadata"][f"{split}_img_concepts"]
    assert len(labels) == len(data["preprocessed_eeg_data"]), f"{split}: EEG/label length mismatch"

if "args" in train:
    args = train["args"]
    for key, value in (args.items() if isinstance(args, dict) else vars(args).items()):
        print(f"{key}: {value}")
if "times" in train:
    times = np.asarray(train["times"])
    print("epoch:", times[[0,-1]], "seconds | samples:", len(times))
    if len(times) > 1: print("sampling rate:", 1 / np.median(np.diff(times)), "Hz")
print("channels:", eeg.shape[2])
assert eeg.shape[2:] == eeg_test.shape[2:], "Train/test EEG dimensions differ"

## 1. Frozen CLIP text prototypes

Default prompts: `a photo of a {concept}`. Unique IDs such as `bat1` and `bat2` remain separate classes. For optional sense-aware prompts, place the THINGS table at `WORK / "things_concepts.tsv"` **before running this section**; it must have `uniqueID`/`conceptID`, `word`, and optionally a definition or synset column. Missing senses fall back to the unique ID; this is not guaranteed disambiguation.

In [ ]:
# === Step A: rep-averaged EEG + frozen CLIP text prototypes (sense-aware if available) ===
import os, numpy as np, torch
import torch.nn.functional as F

if "_clip" not in globals():
    from transformers import CLIPModel, CLIPProcessor
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    _clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE).eval()
    _proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

@torch.no_grad()
def encode_text(texts, batch=64):
    out = []
    for i in range(0, len(texts), batch):
        tok = _proc(text=texts[i:i+batch], return_tensors="pt", padding=True, truncation=True).to(DEVICE)
        t   = _clip.text_model(input_ids=tok["input_ids"], attention_mask=tok.get("attention_mask"))
        emb = F.normalize(_clip.text_projection(t.pooler_output), dim=-1)
        out.append(emb.cpu().numpy())
    return np.concatenate(out, 0)

def clean(c):
    c = str(c)
    if "_" in c and c.split("_")[0].isdigit(): c = "_".join(c.split("_")[1:])
    return c.replace("_", " ").strip()

def to_uid(folder):                      # '00079_bat1'->'bat1', '00001_aircraft_carrier'->'aircraft_carrier'
    p = str(folder).split("_")
    return "_".join(p[1:] if p[0].isdigit() else p)

# Optional: put a tab-separated THINGS concept table in WORK / "things_concepts.tsv" before running this cell.
gloss = {}
if (WORK / "things_concepts.tsv").exists():
    import pandas as pd
    df = pd.read_csv(WORK / "things_concepts.tsv", sep="\t")
    uid_col  = next(c for c in df.columns if c.lower().replace(" ","") in ("uniqueid","conceptid"))
    word_col = next(c for c in df.columns if c.lower() == "word")
    def_col  = next((c for c in df.columns if "definition" in c.lower() or "synset" in c.lower()), None)
    for _, r in df.iterrows():
        word = str(r[word_col]).strip()
        defi = str(r[def_col]).split(";")[0].strip() if def_col else ""
        gloss[str(r[uid_col]).strip()] = f"{word}, {defi}" if defi and defi != "nan" else word

# the substitution happens HERE, before CLIP: "a photo of a bat, nocturnal flying mammal"
prompt = lambda uid: f"a photo of a {gloss.get(uid, uid.replace('_',' '))}"

Xtr = train["preprocessed_eeg_data"].mean(1).astype(np.float32)
Xte = test ["preprocessed_eeg_data"].mean(1).astype(np.float32)

# class key = uniqueID, so bat1 and bat2 stay DISTINCT classes (no merging)
tr_uid = [to_uid(c) for c in train["img_metadata"]["train_img_concepts"]]
te_uid = [to_uid(c) for c in test ["img_metadata"]["test_img_concepts"]]
train_vocab = sorted(set(tr_uid)); test_vocab = sorted(set(te_uid))

Ztr_text = torch.tensor(encode_text([prompt(u) for u in train_vocab]), dtype=torch.float32)
Zte_text = torch.tensor(encode_text([prompt(u) for u in test_vocab ]), dtype=torch.float32)

tr_map = {c:i for i,c in enumerate(train_vocab)}; te_map = {c:i for i,c in enumerate(test_vocab)}
tr_idx = np.array([tr_map[c] for c in tr_uid]); te_idx = np.array([te_map[c] for c in te_uid])

cov = np.mean([u in gloss for u in set(tr_uid)|set(te_uid)]) if gloss else 0.0
print("prompts:", "SENSE-AWARE" if gloss else "plain (provide things_concepts.tsv for sense-aware)", f"| coverage {cov:.0%}")
print("train EEG:", Xtr.shape, "| concepts:", len(train_vocab))
print("test  EEG:", Xte.shape, "| concepts:", len(test_vocab))
print("overlap:", len(set(train_vocab) & set(test_vocab)), "(want 0) | CLIP dim:", Ztr_text.shape[1])
assert not set(train_vocab) & set(test_vocab), "Zero-shot train/test concepts must be disjoint"
N_CH = Xtr.shape[1]
EMB_DIM = Ztr_text.shape[1]

## 2. Label and homonym diagnostics

In [ ]:
# === Verify homonyms become distinct CLIP texts (and embeddings) ===
import numpy as np, re, torch
from collections import defaultdict

raw_all = list(train["img_metadata"]["train_img_concepts"]) + \
          list(test ["img_metadata"]["test_img_concepts"])

base   = lambda u: re.sub(r"(?<=[a-z])\d+$", "", u)        # 'bat1' -> 'bat'
groups = defaultdict(set)
for c in raw_all:
    groups[base(to_uid(c))].add(to_uid(c))
pairs = {b: sorted(s) for b, s in groups.items() if len(s) > 1}

if not pairs:
    print("No multi-sense groups in this subject's concept set.")
else:
    print(f"{len(pairs)} homonym groups. Examples:")
    for b, ids in list(pairs.items())[:8]:
        print("  ", b, "->", ids)

    target = "bat" if "bat" in pairs else next(iter(pairs))
    ids = pairs[target]
    print(f"\n=== '{target}' : {ids} ===")
    for u in ids:
        print(f"  {u:12s} -> {prompt(u)!r}")

    texts = [prompt(u) for u in ids]
    print("\nall prompts distinct strings? ", len(set(texts)) == len(texts))

    fixed  = encode_text(texts)                              # sense-aware
    broken = encode_text([f"a photo of a {u}" for u in ids]) # what you'd get without the gloss
    print("\nsense-aware embeddings — pairwise cosine:")
    print(np.round(fixed @ fixed.T, 3))
    print("un-fixed 'a photo of a bat1/bat2' — pairwise cosine:")
    print(np.round(broken @ broken.T, 3))

## 3. Inspect the CLIP text space

Nearest neighbours use the original embedding space. t-SNE and HDBSCAN islands are exploratory visual summaries and do not demonstrate EEG decoding performance.

In [ ]:
import numpy as np

Z = Ztr_text.numpy()                 # (1654, 512), already unit-norm
sim = Z @ Z.T                        # cosine similarity
np.fill_diagonal(sim, -1.0)          # drop self-matches

# seeds that exist in your cleaned vocab; fall back to random if missing
wanted = ["dog","cat","airplane","guitar","banana","hammer","sofa","shark","piano","truck"]
seeds  = [s for s in wanted if s in train_vocab]
if len(seeds) < 5:
    rng = np.random.default_rng(0)
    seeds += [train_vocab[i] for i in rng.choice(len(train_vocab), 5, replace=False)]

for s in seeds:
    i  = train_vocab.index(s)
    nn = np.argsort(-sim[i])[:5]
    print(f"{s:12s} -> " + ", ".join(f"{train_vocab[j]} ({sim[i,j]:.2f})" for j in nn))

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import numpy as np

Z  = Ztr_text.numpy()
Z2 = TSNE(n_components=2, metric="cosine", init="pca",
          perplexity=30, random_state=0).fit_transform(Z)

plt.figure(figsize=(14, 10))
plt.scatter(Z2[:, 0], Z2[:, 1], s=8, alpha=0.35, color="steelblue")

# 1654 labels is unreadable — annotate a random subset
rng = np.random.default_rng(0)
for i in rng.choice(len(train_vocab), 60, replace=False):
    plt.text(Z2[i, 0], Z2[i, 1], train_vocab[i], fontsize=7)

plt.title("CLIP text embeddings of THINGS concepts (t-SNE, cosine)")
plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# === Detect concept "islands" with density-based clustering ===
import numpy as np, matplotlib.pyplot as plt, matplotlib.cm as cm
from sklearn.manifold import TSNE
from sklearn.cluster import HDBSCAN

Z  = Ztr_text.numpy()
Z2 = TSNE(n_components=2, metric="cosine", init="pca",
          perplexity=30, random_state=0).fit_transform(Z)     # same layout as your plot

# cluster on the 2-D layout -> catches the visual islands; -1 = loose point (noise)
lab   = HDBSCAN(min_cluster_size=8, min_samples=4).fit(Z2).labels_
n_isl = lab.max() + 1
print(f"detected {n_isl} islands | {int(np.sum(lab==-1))} loose points")

gl   = gloss if "gloss" in globals() else {}
name = lambda i: gl.get(train_vocab[i], train_vocab[i].replace("_", " ")).split(",")[0]
pal  = cm.tab20.colors

plt.figure(figsize=(15, 11))
plt.scatter(Z2[lab==-1, 0], Z2[lab==-1, 1], s=6, alpha=0.12, color="lightgray")  # noise
for c in range(n_isl):
    idx = np.where(lab == c)[0]
    plt.scatter(Z2[idx, 0], Z2[idx, 1], s=20, alpha=0.9, color=pal[c % len(pal)])
    cen = Z2[idx].mean(0)                                    # tag island at its centroid
    central = idx[np.argsort(np.linalg.norm(Z2[idx]-cen, axis=1))[:2]]
    plt.text(cen[0], cen[1], ", ".join(name(i) for i in central),
             fontsize=8, fontweight="bold", ha="center")
plt.title(f"Concept islands (HDBSCAN on t-SNE) — {n_isl} islands")
plt.axis("off"); plt.tight_layout(); plt.show()

# full membership, largest island first
for c in sorted(range(n_isl), key=lambda c: -int(np.sum(lab == c))):
    members = [name(i) for i in np.where(lab == c)[0]]
    print(f"island {c:2d} (n={len(members):3d}): {', '.join(members[:25])}"
          + (" ..." if len(members) > 25 else ""))

## 4. EEG encoder and concept-disjoint split

Infer channels and embedding width from data. Use spatial mixing, temporal convolutions, pooling and a normalized projection. Hold out 200 training concepts before fitting normalization; neither validation nor test EEG contributes to normalization.

All EEG tensors are kept on the selected device. This workflow is intended for a GPU with ample memory; reduce training `BS` if necessary. Evaluation is batched.

In [ ]:
# === Setup: standardize EEG, move everything to GPU once, define the encoder ===
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np

class EEGEncoder2(nn.Module):
    def __init__(self, n_ch=N_CH, emb_dim=EMB_DIM, h=128, p=0.4):
        super().__init__()
        self.spatial  = nn.Sequential(nn.Conv1d(n_ch, h, 1), nn.BatchNorm1d(h), nn.GELU())
        self.temporal = nn.Sequential(
            nn.Conv1d(h, h, 11, padding=5),     nn.BatchNorm1d(h),   nn.GELU(), nn.AvgPool1d(2),
            nn.Conv1d(h, 2*h, 11, padding=5),   nn.BatchNorm1d(2*h), nn.GELU(), nn.AvgPool1d(2),
            nn.Conv1d(2*h, 2*h, 11, padding=5), nn.BatchNorm1d(2*h), nn.GELU(),
            nn.AdaptiveAvgPool1d(4))
        self.head = nn.Sequential(nn.Flatten(), nn.Dropout(p), nn.Linear(2*h*4, emb_dim))
    def forward(self, x):
        return F.normalize(self.head(self.temporal(self.spatial(x))), dim=-1)

# Split by concept before fitting normalization to exclude held-out validation EEG.
rng = np.random.default_rng(SEED)
nC = len(train_vocab)
assert nC > 200, "Need more than 200 training concepts for this protocol"
val_set = set(rng.choice(nC, 200, replace=False).tolist())
keep_g = np.array([c for c in range(nC) if c not in val_set])
val_g = np.array(sorted(val_set))
keep_map = {g: i for i, g in enumerate(keep_g)}
val_map = {g: i for i, g in enumerate(val_g)}
img_val = np.isin(tr_idx, list(val_set))
tr_i, va_i = np.where(~img_val)[0], np.where(img_val)[0]
_mu = Xtr[tr_i].mean((0, 2), keepdims=True)
_sd = Xtr[tr_i].std((0, 2), keepdims=True) + 1e-6
Xtr_g = torch.tensor((Xtr - _mu) / _sd, dtype=torch.float32, device=DEVICE)
Xte_g = torch.tensor((Xte - _mu) / _sd, dtype=torch.float32, device=DEVICE)
ytr_g = torch.tensor(tr_idx, dtype=torch.long, device=DEVICE)
Ptr, Pte = Ztr_text.to(DEVICE), Zte_text.to(DEVICE)

@torch.no_grad()
def score(encoder, Xg, idx, P, k=5, batch_size=256):
    encoder.eval()
    predictions = []
    for start in range(0, len(Xg), batch_size):
        similarities = encoder(Xg[start:start + batch_size]) @ P.T
        predictions.append(similarities.topk(min(k, len(P)), dim=1).indices.cpu().numpy())
    top = np.concatenate(predictions)
    return float(np.mean(top[:, 0] == idx)), float(np.mean(np.any(top == np.asarray(idx)[:, None], axis=1)))

print("EEG:", tuple(Xtr_g.shape), "| device:", DEVICE)
print("Normalization fitted on", len(tr_i), "optimization examples only")

### Optional baseline

Enabled with `RUN_BASELINE`. This comparison trains on all training concepts and reports training fit only. The regularized model below starts fresh; baseline weights are never reused.

In [ ]:
if RUN_BASELINE:
    # === Baseline training: stronger encoder, full data on GPU, big batch, many epochs ===
    model = EEGEncoder2().to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=2e-4)
    EPOCHS, BS, logit_scale = 150, 1024, 1/0.07
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

    N = Xtr_g.shape[0]
    print("params:", round(sum(p.numel() for p in model.parameters())/1e6,2), "M | device:", DEVICE)
    for ep in range(1, EPOCHS+1):
        model.train(); perm = torch.randperm(N, device=DEVICE); ls=correct=0
        for i in range(0, N, BS):
            b = perm[i:i+BS]
            logits = logit_scale * (model(Xtr_g[b]) @ Ptr.T)
            loss = F.cross_entropy(logits, ytr_g[b])
            opt.zero_grad(); loss.backward(); opt.step()
            ls += loss.item()*len(b); correct += (logits.argmax(1)==ytr_g[b]).sum().item()
        sched.step()
        if ep==1 or ep%10==0:
            t1,t5 = score(model, Xtr_g, tr_idx, Ptr)
            print(f"ep {ep:3d} | loss {ls/N:.3f} | train top1 {correct/N:5.1%} | train eval top1 {t1:5.1%} top5 {t5:5.1%}")
else:
    print("Baseline skipped; set RUN_BASELINE = True to compare training fit.")

## 5. Regularized training and model selection

Train with dropout, weight decay, Gaussian input noise and a cosine learning-rate schedule. Select the checkpoint by held-out-concept validation top-5, with patience 20. Evaluate the selected checkpoint once on the test vocabulary.

The checkpoint, normalization, vocabulary, prompts, and metrics are written under `WORK / "results"`. Fixed seeds help repeat runs, but GPU operations can still be nondeterministic.

In [ ]:
Xk = Xtr_g[torch.tensor(tr_i, device=DEVICE)]
yk = torch.tensor([keep_map[c] for c in tr_idx[tr_i]], dtype=torch.long, device=DEVICE)
Xv = Xtr_g[torch.tensor(va_i, device=DEVICE)]
yv = np.array([val_map[c] for c in tr_idx[va_i]])
Pk = Ztr_text.to(DEVICE)[torch.tensor(keep_g, device=DEVICE)]
Pv = Ztr_text.to(DEVICE)[torch.tensor(val_g,  device=DEVICE)]
print(f"train: {len(tr_i)} imgs / {len(keep_g)} concepts | val: {len(va_i)} imgs / {len(val_g)} held-out concepts")

model = EEGEncoder2(p=0.5).to(DEVICE)                      # more dropout
opt   = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=2e-3)  # more weight decay
EPOCHS, BS, logit_scale = 120, 1024, 1/0.07
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

best, best_state, patience, bad, N = -1, None, 20, 0, Xk.shape[0]
for ep in range(1, EPOCHS+1):
    model.train(); perm = torch.randperm(N, device=DEVICE)
    for i in range(0, N, BS):
        b = perm[i:i+BS]
        xb = Xk[b] + 0.1*torch.randn_like(Xk[b])          # noise augmentation
        loss = F.cross_entropy(logit_scale*(model(xb)@Pk.T), yk[b])
        opt.zero_grad(); loss.backward(); opt.step()
    sched.step()
    v1, v5 = score(model, Xv, yv, Pv)
    if v5 > best:
        best, best_state, bad = v5, {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}, 0
        best_epoch = ep
    else: bad += 1
    if ep==1 or ep%5==0: print(f"ep {ep:3d} | val zs top1 {v1:5.1%} top5 {v5:5.1%} | best {best:5.1%}")
    if bad >= patience: print(f"early stop @ ep {ep}"); break

model.load_state_dict(best_state)
t1, t5 = score(model, Xte_g, te_idx, Pte)
print(f"\nBEST MODEL (selected on val) -> TEST zero-shot: top1 {t1:.1%}  top5 {t5:.1%}  (chance {1/len(test_vocab):.1%}/{min(5, len(test_vocab))/len(test_vocab):.1%})")
zero_shot_model = model
metrics = {"subject": SUB, "seed": SEED, "test_candidates": len(test_vocab),
           "top1": t1, "top5": t5, "best_validation_top5": float(best),
           "selected_epoch": best_epoch, "stopped_epoch": ep, "prompt_mode": "sense-aware" if gloss else "plain"}
torch.save({"model_state_dict": zero_shot_model.state_dict(), "n_ch": N_CH, "emb_dim": EMB_DIM,
            "normalization_mean": _mu, "normalization_std": _sd,
            "train_vocab": train_vocab, "test_vocab": test_vocab, "seed": SEED,
            "test_prompts": [prompt(u) for u in test_vocab]}, RESULTS / "zero_shot_model.pt")
(RESULTS / "metrics.json").write_text(json.dumps(metrics, indent=2))
print("Saved model and metrics to:", RESULTS.resolve())

## 6. Zero-shot evaluation and inspection

In [ ]:
# === Inspect: random test EEG -> true vs predicted concept (zero-shot, 200-way) ===
import numpy as np, torch
zero_shot_model.eval()
P = Zte_text.to(DEVICE)                       # (200, 512) test-concept prototypes

picks = np.random.default_rng(7).choice(len(te_idx), size=10, replace=False)
with torch.no_grad():
    z    = zero_shot_model(Xte_g[torch.tensor(picks, device=DEVICE)])   # (10, 512)
    top3 = (z @ P.T).topk(3, dim=1).indices.cpu().numpy()

print(f"{'TRUE':20s} | {'PREDICTED':20s} | hit | top-3 candidates")
print("-"*88)
for r, i in enumerate(picks):
    true = test_vocab[te_idx[i]]
    pred = test_vocab[top3[r, 0]]
    hit  = "Y " if top3[r,0]==te_idx[i] else ("~ " if te_idx[i] in top3[r] else ". ")
    cand = ", ".join(test_vocab[j] for j in top3[r])
    print(f"{true:20s} | {pred:20s} | {hit}  | {cand}")

### Confusion matrix
Concept indices follow the saved test vocabulary.

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
pred_parts = []
zero_shot_model.eval()
with torch.no_grad():
    for start in range(0, len(Xte_g), 256):
        pred_parts.append((zero_shot_model(Xte_g[start:start+256]) @ Pte.T).argmax(1).cpu().numpy())
pred = np.concatenate(pred_parts)
cmatrix = confusion_matrix(te_idx, pred, labels=np.arange(len(test_vocab)))
np.save(RESULTS / "confusion_matrix.npy", cmatrix)
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cmatrix, cmap="Blues")
ax.set(xlabel="Predicted concept index", ylabel="True concept index", title=f"Zero-shot confusion matrix ({len(test_vocab)} candidates)")
fig.colorbar(im, ax=ax, label="Number of examples")
fig.tight_layout(); fig.savefig(RESULTS / "confusion_matrix.png", dpi=150); plt.show()
print("Index order: test_vocab; saved in the checkpoint.")

## Optional experiment: within-concept exemplar generalization

Enable `RUN_WITHIN_CONCEPT` in Section 0. Hold out two images per training concept and fit an independent model and normalization. Every evaluated concept is seen during training; this answers a different question from zero-shot classification. The zero-shot model remains available.

In [ ]:
if RUN_WITHIN_CONCEPT:
    # === Within-concept eval: hold out 2 images/concept (exemplar generalization, NOT zero-shot) ===
    import numpy as np, torch, torch.nn.functional as F

    rng = np.random.default_rng(0)
    eval_mask = np.zeros(len(tr_idx), dtype=bool)
    for c in range(len(train_vocab)):
        imgs = np.where(tr_idx == c)[0]
        eval_mask[rng.choice(imgs, size=min(2, len(imgs)), replace=False)] = True
    tr_i, ev_i = np.where(~eval_mask)[0], np.where(eval_mask)[0]
    print(f"train {len(tr_i)} imgs | eval {len(ev_i)} imgs | all {len(train_vocab)} concepts seen in both")

    within_mu = Xtr[tr_i].mean((0, 2), keepdims=True)
    within_sd = Xtr[tr_i].std((0, 2), keepdims=True) + 1e-6
    within_X = torch.tensor((Xtr - within_mu) / within_sd, dtype=torch.float32, device=DEVICE)
    Xtr_f = within_X[torch.tensor(tr_i, device=DEVICE)]
    ytr_f = torch.tensor(tr_idx[tr_i], dtype=torch.long, device=DEVICE)
    Xev_f = within_X[torch.tensor(ev_i, device=DEVICE)]
    yev   = tr_idx[ev_i]
    P_all = Ztr_text.to(DEVICE)                      # 1654-way: candidates are all KNOWN concepts

    within_model = EEGEncoder2(p=0.5).to(DEVICE)
    opt   = torch.optim.AdamW(within_model.parameters(), lr=5e-4, weight_decay=2e-3)
    EPOCHS, BS, logit_scale = 40, 1024, 1/0.07
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    N = Xtr_f.shape[0]
    for ep in range(1, EPOCHS+1):
        within_model.train(); perm = torch.randperm(N, device=DEVICE)
        for i in range(0, N, BS):
            b = perm[i:i+BS]
            xb = Xtr_f[b] + 0.1*torch.randn_like(Xtr_f[b])
            loss = F.cross_entropy(logit_scale*(within_model(xb)@P_all.T), ytr_f[b])
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        if ep==1 or ep%5==0:
            within_model.eval()
            with torch.no_grad(): s = within_model(Xev_f) @ P_all.T
            t1 = (s.argmax(1).cpu().numpy()==yev).mean()
            tk = s.topk(5,1).indices.cpu().numpy()
            t5 = np.mean([yev[i] in tk[i] for i in range(len(yev))])
            print(f"ep {ep:2d} | eval top1 {t1:5.1%} top5 {t5:5.1%}  (1654-way, chance {1/len(train_vocab):.2%})")
else:
    print("Optional within-concept experiment skipped.")

### Repeated 200-candidate within-concept evaluation
Sample 50 candidate subsets. Matching candidate counts does not make this protocol equivalent to testing unseen concepts. Reported variation is the standard deviation over subsets, not a confidence interval.

In [ ]:
if RUN_WITHIN_CONCEPT:
    # === Fair comparison: re-score the trained within_model at 200-way (matches the zero-shot test) ===
    import numpy as np, torch
    within_model.eval()
    P = Ztr_text.to(DEVICE)
    with torch.no_grad():
        sims = (within_model(Xev_f) @ P.T).cpu().numpy()     # (n_eval, 1654)

    rng = np.random.default_rng(0); NWAY, REPEATS, nC = 200, 50, P.shape[0]
    t1s, t5s = [], []
    for _ in range(REPEATS):
        subset = rng.choice(nC, NWAY, replace=False)
        idx = np.where(np.isin(yev, subset))[0]
        sub = sims[np.ix_(idx, subset)]
        pos = {c:i for i,c in enumerate(subset)}
        true_local = np.array([pos[yev[i]] for i in idx])
        order = np.argsort(-sub, axis=1)
        t1s.append(np.mean(order[:,0]==true_local))
        t5s.append(np.mean([true_local[r] in order[r,:5] for r in range(len(idx))]))
    print(f"200-way within-concept: top1 {np.mean(t1s):.1%} ± {np.std(t1s):.1%} | "
          f"top5 {np.mean(t5s):.1%} ± {np.std(t5s):.1%}  (chance 0.5%)")

## Optional diagnostic: semantic islands

Use the selected **zero-shot model**. The same-island score is a coarse, cluster-dependent diagnostic, not a replacement for exact concept accuracy. Clusters are derived from the t-SNE projection; they are sensitive to layout and hyperparameters.

In [ ]:
# === Tunable-precision islands + prediction inspection ===
import numpy as np, matplotlib.pyplot as plt, matplotlib.cm as cm, torch
from sklearn.manifold import TSNE
from sklearn.cluster import HDBSCAN

# ---------- PRECISION KNOBS ----------
MIN_ISLAND = 6
MERGE      = 2
# -------------------------------------

Z = Ztr_text.numpy()
if "_Z2_isl" not in globals():
    _Z2_isl = TSNE(n_components=2, metric="cosine", init="pca",
                   perplexity=30, random_state=0).fit_transform(Z)
Z2 = _Z2_isl
print("t-SNE coord std (MERGE is in these units):", Z2.std(0).round(1))

lab = HDBSCAN(min_cluster_size=MIN_ISLAND, min_samples=3,
              cluster_selection_epsilon=float(MERGE)).fit(Z2).labels_

gl   = gloss if "gloss" in globals() else {}
name = lambda i: gl.get(train_vocab[i], train_vocab[i].replace("_", " ")).split(",")[0]

real = [c for c in set(lab) if c != -1]
assert real, "all noise — lower MIN_ISLAND or MERGE"
cen2 = {c: Z2[lab == c].mean(0) for c in real}
for i in np.where(lab == -1)[0]:
    lab[i] = min(real, key=lambda c: np.linalg.norm(Z2[i] - cen2[c]))

# island NAME = word whose CLIP embedding is closest to the island's MEAN CLIP vector
island_name = {}
for c in set(lab):
    idx  = np.where(lab == c)[0]
    mean = Z[idx].mean(0); mean /= np.linalg.norm(mean)      # average CLIP vector of the island
    best = idx[np.argmax(Z[idx] @ mean)]                     # member closest to that average (cosine)
    island_name[c] = name(best)
print(f"{len(set(lab))} islands  (MIN_ISLAND={MIN_ISLAND}, MERGE={MERGE})")

# ---------- plot at this precision ----------
pal = cm.tab20.colors
plt.figure(figsize=(15, 11))
for c in sorted(set(lab)):
    idx = np.where(lab == c)[0]
    plt.scatter(Z2[idx, 0], Z2[idx, 1], s=18, alpha=0.85, color=pal[c % len(pal)])
    plt.text(*Z2[idx].mean(0), island_name[c], fontsize=8, fontweight="bold", ha="center")
plt.title(f"{len(set(lab))} islands | MIN_ISLAND={MIN_ISLAND}, MERGE={MERGE}")
plt.axis("off"); plt.tight_layout(); plt.show()

# ---------- map TEST concepts onto these islands (nearest centroid in CLIP space) ----------
isl_ids = np.array(sorted(set(lab)))
C = np.stack([Z[lab == c].mean(0) for c in isl_ids]); C /= np.linalg.norm(C, axis=1, keepdims=True)
concept_isl = isl_ids[(Zte_text.numpy() @ C.T).argmax(1)]

zero_shot_model.eval()
with torch.no_grad():
    pred = np.concatenate([(zero_shot_model(Xte_g[i:i+256]) @ Pte.T).argmax(1).cpu().numpy() for i in range(0, len(Xte_g), 256)])

true_isl, pred_isl = concept_isl[te_idx], concept_isl[pred]
print(f"\nexact top-1: {(pred==te_idx).mean():.1%}  |  same-island top-1: {(true_isl==pred_isl).mean():.1%}")

tn = lambda v: gl.get(v, v.replace("_", " ")).split(",")[0]
picks = np.random.default_rng(7).choice(len(te_idx), size=14, replace=False)
print(f"\n{'TEXT':15s} | {'TRUE ISLAND':20s} | {'PREDICTED ISLAND':20s} | same")
print("-" * 70)
for i in picks:
    print(f"{tn(test_vocab[te_idx[i]]):15s} | {island_name[true_isl[i]]:20s} | "
          f"{island_name[pred_isl[i]]:20s} | {'Y' if true_isl[i]==pred_isl[i] else '.'}")

## Optional appendix: DNN feature-map inventory
Enable `RUN_DNN_EXPLORATION` to list the separate OSF feature-map component; it is not used for training.

In [ ]:
if RUN_DNN_EXPLORATION:
    # === List the DNN feature-maps component (wnbm3) ===
    import requests, numpy as np, os, zipfile
    DNN_NODE = "wnbm3"

    def _children(url):
        items=[]
        while url:
            js=requests.get(url,timeout=60).json(); items+=js.get("data",[]); url=js.get("links",{}).get("next")
        return items
    def _root(node):
        prov=_children(f"https://api.osf.io/v2/nodes/{node}/files/")
        return next(p for p in prov if p["attributes"]["provider"]=="osfstorage")["relationships"]["files"]["links"]["related"]["href"]
    def _walk(url, pre=""):
        out=[]
        for it in _children(url):
            a=it["attributes"]; p=f"{pre}/{a['name']}".strip("/")
            if a["kind"]=="file": out.append((p, a.get("size"), it["links"]["download"]))
            else: out += _walk(it["relationships"]["files"]["links"]["related"]["href"], p)
        return out

    files = _walk(_root(DNN_NODE))
    print(f"{len(files)} files:")
    for p,s,_ in sorted(files): print(f"  {p}  ({(s or 0)/1e6:.0f} MB)")